# KubeMedic GRPO Training

Self-contained version of the KubeMedic single-session GRPO trainer.

Run cells top to bottom. Designed for a GPU runtime (Colab T4/L4/A100, HF Spaces with GPU, etc.). On CPU the smoke test still works but training will be slow / unsupported by 4-bit quantization.

## 1. Install dependencies

Single install cell. Pulls the KubeMedic env client plus the training stack we actually use: `transformers`, `trl`, `peft`, `accelerate`, `datasets`, `bitsandbytes`, `unsloth`, the matched `torch` trio, and `matplotlib`/`pandas`/`seaborn` for plotting. **Auto-restarts the kernel at the end** — that is intentional and required so the freshly installed `transformers`/`torch` are loaded cleanly.

In [2]:

SPACE_ID = 'ashiqabdulkhader/Kubemedic'
SPACE_URL = f'https://huggingface.co/spaces/{SPACE_ID}.git'
KUBEMEDIC_SRC = '/tmp/kubemedic-src'

In [4]:
import os, shutil, subprocess



if os.path.isdir(KUBEMEDIC_SRC):
    shutil.rmtree(KUBEMEDIC_SRC)
print('Cloning Space:', SPACE_URL)
subprocess.check_call(['git', 'clone', '--depth', '1', SPACE_URL, KUBEMEDIC_SRC])

# Install only what this notebook actually needs. Why each step exists:
#
#   1) Hard-uninstall torch trio + vllm + xformers. Past runs (or the runtime
#      base image, e.g. HF Spaces / Colab) often leave torch 2.11 + vllm 0.18
#      + xformers behind. With those still on disk, `pip install --force-
#      reinstall torch==2.10.0` will SUCCEED but pip's resolver then keeps
#      torch 2.11 because it satisfies vllm/xformers' loose torch>=… deps,
#      and the eventual error is "torchvision 0.25 requires torch==2.10.0,
#      but you have torch 2.11.0". A clean uninstall first sidesteps that.
#
#   2) Kubemedic env client without [train] extras (skips wandb + vllm +
#      plot deps; we install plot deps explicitly below).
#
#   3) Training stack pinned to a coherent set: transformers<5 (peft 0.x can't
#      import in transformers v5), trl<=0.24 (trl 1.x requires transformers
#      v5), peft<0.18 (latest unsloth 2026.4 wants peft>=0.18, but that
#      cascades to other breakages — we'll pin unsloth to a peft<0.18-friendly
#      version below). transformers tightened to 4.57.1–4.57.3 so it falls
#      inside unsloth 2025.12.10's allowed range.
#
#   4) torch trio with --no-deps so pip can't re-resolve and keep torch 2.11.
#
#   5) unsloth==2025.12.10 + unsloth_zoo==2026.1.1: the most recent pair that
#      still allows peft<0.18 and torch>=2.4 (anything 2026.1.x or newer
#      requires peft>=0.18). --no-deps so they don't pull torchao/xformers/
#      diffusers/tyro that we don't use; you'll see harmless warnings about
#      those, ignore them.
#
#   6) Plotting + numerics for the local dashboards we generate at the end.

%pip uninstall -q -y torch torchvision torchaudio vllm xformers
%pip install -q -U "{KUBEMEDIC_SRC}"
%pip install -q --force-reinstall \
    "transformers" \
    "trl" \
    "peft" \
    "accelerate" \
    "datasets" \
    "bitsandbytes" \
    "hf_transfer"
%pip install -q --no-deps --force-reinstall \
    "torch==2.10.0" "torchvision==0.25.0" "torchaudio==2.10.0"
%pip install -q --no-deps --force-reinstall \
    "unsloth==2025.12.10" "unsloth_zoo==2026.1.1"
%pip install -q -U "numpy>=" "matplotlib" "pandas" "seaborn"

_cache_dir = os.path.expanduser('~/unsloth_compiled_cache')
if os.path.isdir(_cache_dir):
    shutil.rmtree(_cache_dir, ignore_errors=True)
    print('Cleared ~/unsloth_compiled_cache so GRPO patches recompile against pinned trl.')

print('\nInstall complete. Restarting kernel so transformers/torch/peft reload cleanly...')
print('After the kernel restarts, re-run from the *next* cell — do NOT re-run this install cell.\n')

# transformers, torch, and peft were force-reinstalled on disk, but the running
# kernel still has the *old* modules in sys.modules. Re-importing without a
# restart raises misleading transformers lazy-loader errors. Auto-restart so
# this isn't optional.
try:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
except Exception:
    import os, signal
    os.kill(os.getpid(), signal.SIGTERM)

## 2. Imports and environment setup

Loads `HF_TOKEN` (from `.env`, Colab `userdata`, or the `hf` CLI), sets matplotlib to a headless backend, and imports the training stack.

In [ ]:
import argparse
import csv
import json
import math
import os
import random
import shlex
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from typing import Any
try:
    from unsloth import FastLanguageModel
    UNSLOTH_AVAILABLE = True
    try:
        from unsloth import FastModel
    except ImportError:
        FastModel = None
    try:
        from unsloth import PatchFastRL
        PatchFastRL('GRPO', FastLanguageModel)
        print('unsloth loaded; GRPO patches applied via PatchFastRL.')
    except ImportError:
        print('unsloth loaded; relying on unsloth_zoo auto-patches (no PatchFastRL).')
    except Exception as patch_exc:
        print(f'unsloth loaded; PatchFastRL skipped ({patch_exc}); auto-patches still active.')
except Exception as exc:
    print(f'unsloth unavailable ({exc}); falling back to standard transformers loading.')
    FastLanguageModel = None
    FastModel = None
    UNSLOTH_AVAILABLE = False

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'kubemedic-mpl'))
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('TRL_EXPERIMENTAL_SILENCE', '1')
os.environ['WANDB_DISABLED'] = 'true'  # this notebook does not use wandb at all
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'  # avoid hard dependency on hf_transfer

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
env_path = next((path for path in env_candidates if path.exists()), None)
if env_path:
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)

if not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass

if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = subprocess.check_output(['hf', 'auth', 'token'], text=True).strip()
    except Exception:
        pass



import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import transformers
from datasets import Dataset
from packaging.version import Version
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer

# PyPI `trl` 0.18–0.24 does not include `trl.experimental.openenv` (Meta's OpenEnv
# bridge lives on a different install / main-branch build). The rollout cell
# below defines `generate_rollout_completions_local` with the same return
# shape; we bind it when the import is missing.
try:
    from trl.experimental.openenv import generate_rollout_completions
except ImportError:
    generate_rollout_completions = None  # set in rollout cell after local def

from Kubemedic import KubemedicAction, KubemedicEnv, KubemedicObservation
from Kubemedic.models import ToolName

try:
    from websockets.exceptions import ConnectionClosedError, ConnectionClosedOK
except Exception:
    ConnectionClosedError = Exception
    ConnectionClosedOK = Exception

print('Kernel python   :', sys.executable)
print('Torch version   :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
print('Unsloth         :', UNSLOTH_AVAILABLE)
print('HF token loaded :', bool(os.environ.get('HF_TOKEN')))

## 3. Configuration

Hyperparameters, scenario list, system prompt, output paths. Edit any of these knobs in this single cell.

In [4]:
ENV_URL = f"https://{SPACE_ID.replace('/', '-')}.hf.space"
MODEL_ID = 'unsloth/Qwen3.5-2B'
OUTPUT_DIR = Path('outputs/kubemedic-qwen3-1.7b-grpo')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_EPISODES = 16  # 2 GRPO groups per step at batch_size=8
EVAL_EPISODES = 2
NUM_EPOCHS = 5
NUM_GENERATIONS = 4  # tighter GRPO advantage estimates than n=2
PER_DEVICE_TRAIN_BATCH_SIZE = 8  # 2 * NUM_GENERATIONS
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-5
MAX_COMPLETION_LENGTH = 512
MAX_TURNS = 8
SAVE_STEPS = 4
LOGGING_STEPS = 1
SEED = 42
# vLLM fast-inference requires 16-bit weights, which is incompatible with the
# 4-bit QLoRA path below, so leave it off here. Flip back on (and switch to a
# bf16 model) only if you have the VRAM headroom for it.
VLLM_MODE = 'colocate'
VLLM_GPU_MEMORY_UTILIZATION = 0.5
USE_4BIT = True   # required for the *-unsloth-bnb-4bit repo
USE_VLLM = False  # incompatible with USE_4BIT=True
USE_UNSLOTH = True  # 2x faster + lower VRAM; auto-falls back if unavailable
USE_GRADIENT_CHECKPOINTING = True  # 4-bit run is VRAM-sensitive; GC pays for itself
USE_FLASH_ATTENTION = True  # SDPA backend, Flash Attention 2 when available
MAX_SEQ_LENGTH = 2048
LORA_RANK = 16
SKIP_SMOKE_TEST = False
SMOKE_ONLY = False
DEFAULT_SCENARIOS = ['KUBE-01', 'KUBE-03', 'KUBE-04', 'KUBE-05', 'KUBE-06']
SCENARIOS = list(DEFAULT_SCENARIOS)

CONNECT_TIMEOUT_S = 30.0
MESSAGE_TIMEOUT_S = 240.0
VALID_TOOLS = set(ToolName.__args__)  # type: ignore[attr-defined]

SYSTEM_PROMPT = """You are a Kubernetes SRE agent operating KubeMedic.
Diagnose the cluster carefully, use the available actions, minimize blast radius, and stop once the cluster is healthy.

Respond with exactly one action per turn using this format:
- TOOL_NAME key=value key=value
- TOOL_NAME {\"json\": \"object\"}

Allowed tool names:
kubectl_get
kubectl_describe
kubectl_logs
kubectl_top_pods
kubectl_top_nodes
kubectl_patch_resources
kubectl_patch_tolerations
kubectl_cordon
kubectl_uncordon
kubectl_delete_pod
kubectl_delete_workload

Rules:
- Never include markdown fences or explanations.
- Prefer inspect-first behavior before mutating resources.
- Use namespace=challenge unless the observation clearly requires something else.
"""

TRAINING_HINT = (
    'Diagnose and repair this Kubernetes incident. Output one action only, in the required syntax.'
)

print('ENV_URL   :', ENV_URL)
print('MODEL_ID  :', MODEL_ID)
print('OUTPUT_DIR:', OUTPUT_DIR.resolve())
print('SCENARIOS :', SCENARIOS)

## 4. Helper functions

Observation formatting, action parsing, prompt rendering, retry/connection helpers, and rollout utilities. These are the pure-Python helpers from the script.

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_env(env_url: str):
    return KubemedicEnv(
        base_url=env_url,
        connect_timeout_s=CONNECT_TIMEOUT_S,
        message_timeout_s=MESSAGE_TIMEOUT_S,
    ).sync()


def format_observation(obs: KubemedicObservation) -> str:
    pods = []
    for pod in obs.pods:
        parts = [f'phase={pod.phase}']
        if pod.reason:
            parts.append(f'reason={pod.reason}')
        if pod.restarts:
            parts.append(f'restarts={pod.restarts}')
        pods.append(f'- {pod.name} ({pod.namespace}): ' + ' '.join(parts))

    nodes = []
    for node in obs.nodes:
        pressure = [c.type for c in node.conditions if c.status == 'True' and c.type.endswith('Pressure')]
        nodes.append(
            f"- {node.name}: ready={node.ready} pressure={','.join(pressure) if pressure else 'none'}"
        )

    info = obs.info or {}
    reward_breakdown = info.get('reward_breakdown', {})
    breakdown_text = ', '.join(f'{k}={v:.2f}' for k, v in reward_breakdown.items()) if reward_breakdown else 'none'

    return (
        f'{TRAINING_HINT}\n\n'
        f'Scenario: {obs.scenario}\n'
        f'Time step: {obs.t}\n'
        f"Blocked reason: {obs.blocked_reason or 'none'}\n"
        f'Last reward: {float(obs.reward or 0.0):.2f}\n'
        f"Disruptions: {info.get('disruptions', 0)}\n"
        f'Reward breakdown: {breakdown_text}\n\n'
        f"Pods:\n{chr(10).join(pods) if pods else '- none'}\n\n"
        f"Nodes:\n{chr(10).join(nodes) if nodes else '- none'}"
    )


def format_history(history: list[dict[str, Any]]) -> str:
    if not history:
        return ''
    lines = ['Previous actions:']
    for item in history:
        output = item['output']
        if len(output) > 220:
            output = output[:220] + '...'
        lines.append(f"- {item['action']} -> reward={item['reward']:.2f} | {output}")
    return '\n'.join(lines)


def coerce_value(value: str) -> Any:
    lowered = value.lower()
    if lowered == 'true':
        return True
    if lowered == 'false':
        return False
    if lowered in ('none', 'null'):
        return None
    try:
        return json.loads(value)
    except Exception:
        pass
    try:
        return int(value)
    except ValueError:
        pass
    return value


def parse_action_text(text: str) -> KubemedicAction | None:
    cleaned = text.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.strip('`').strip()
    line = cleaned.splitlines()[0].strip() if cleaned else ''
    if not line:
        return None

    parts = line.split(maxsplit=1)
    tool = parts[0].strip()
    if tool not in VALID_TOOLS:
        return None

    if len(parts) == 1:
        return KubemedicAction(tool=tool, args={})

    remainder = parts[1].strip()
    if not remainder:
        return KubemedicAction(tool=tool, args={})

    args: dict[str, Any] = {}
    if remainder.startswith('{'):
        parsed = json.loads(remainder)
        if not isinstance(parsed, dict):
            return None
        args = parsed
    else:
        for token in shlex.split(remainder):
            if '=' not in token:
                continue
            key, value = token.split('=', 1)
            args[key] = coerce_value(value)

    return KubemedicAction(tool=tool, args=args)


def render_prompt(tokenizer, messages: list[dict[str, str]]) -> str:
    if getattr(tokenizer, 'chat_template', None):
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
    system_lines = [msg['content'] for msg in messages if msg['role'] == 'system']
    user_lines = [msg['content'] for msg in messages if msg['role'] == 'user']
    return '\n\n'.join(
        [
            *(['SYSTEM:\n' + '\n'.join(system_lines)] if system_lines else []),
            *(['USER:\n' + '\n'.join(user_lines)] if user_lines else []),
            'ASSISTANT:\n',
        ]
    )


def is_connection_error(exc: Exception) -> bool:
    text = str(exc)
    return (
        isinstance(exc, (ConnectionClosedError, ConnectionClosedOK))
        or 'ConnectionClosed' in exc.__class__.__name__
        or 'no close frame received or sent' in text
        or 'close frame' in text
        or 'websocket' in text.lower()
    )


def build_dataset(num_rows: int) -> Dataset:
    rows = [{'prompt': TRAINING_HINT} for _ in range(num_rows)]
    return Dataset.from_list(rows)


def infer_lora_target_modules(model) -> list[str]:
    model_type = getattr(model.config, 'model_type', '')
    if model_type in {'qwen2', 'qwen2_5', 'qwen3'}:
        return ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    if model_type in {'gpt2'}:
        return ['c_attn', 'c_proj', 'c_fc']
    module_names = {name.split('.')[-1] for name, _ in model.named_modules()}
    preferred = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'c_attn', 'c_proj', 'c_fc']
    inferred = [name for name in preferred if name in module_names]
    if inferred:
        return inferred
    raise ValueError(f'Could not infer LoRA target modules for model_type={model_type!r}.')


def reward_total(completions: list[str], **kwargs) -> list[float]:
    rewards = kwargs.get('total_reward') if kwargs else None
    return [float(value) for value in rewards] if rewards else [0.0 for _ in completions]

print('Helpers ready.')

## 5. Rollout utilities

Local generation fallback (used when vLLM is off) and the per-episode rollout that drives the live KubeMedic environment.

In [ ]:
def generate_rollout_completions_local(trainer: GRPOTrainer, prompts: list[str]) -> list[dict[str, Any]]:
    tokenizer = trainer.processing_class
    model = trainer.model
    results: list[dict[str, Any]] = []

    for prompt_text in prompts:
        encoded = tokenizer(prompt_text, return_tensors='pt')
        encoded = {key: value.to(model.device) for key, value in encoded.items()}
        generation = model.generate(
            **encoded,
            do_sample=True,
            temperature=trainer.temperature,
            top_p=trainer.top_p if trainer.top_p is not None else 1.0,
            top_k=trainer.top_k if trainer.top_k is not None else 50,
            max_new_tokens=trainer.max_completion_length,
            return_dict_in_generate=True,
            output_scores=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        sequence = generation.sequences[0]
        prompt_len = encoded['input_ids'].shape[1]
        completion_ids = sequence[prompt_len:]
        if generation.scores:
            transition_scores = model.compute_transition_scores(
                generation.sequences,
                generation.scores,
                normalize_logits=True,
            )[0]
            logprobs = [float(score) for score in transition_scores.tolist()[: len(completion_ids)]]
        else:
            logprobs = []

        results.append(
            {
                'prompt_ids': sequence[:prompt_len].tolist(),
                'completion_ids': completion_ids.tolist(),
                'logprobs': logprobs,
                'text': tokenizer.decode(completion_ids, skip_special_tokens=True),
            }
        )
    return results


def rollout_once(
    trainer: GRPOTrainer,
    env_url: str,
    tokenizer,
    scenario: str,
    max_turns: int,
    max_total_tokens: int,
    max_retries: int = 3,
) -> dict[str, Any]:
    last_exc: Exception | None = None
    for attempt in range(1, max_retries + 1):
        env = make_env(env_url)
        try:
            result = env.reset(scenario=scenario)
            observation = result.observation

            prompt_ids: list[int] = []
            completion_ids: list[int] = []
            logprobs: list[float] = []
            step_rewards: list[float] = []
            history: list[dict[str, Any]] = []
            valid_actions = 0
            total_actions = 0
            tool_usage: dict[str, int] = {}
            sample_completion = ''

            for _ in range(max_turns):
                if result.done or len(completion_ids) >= max_total_tokens:
                    break

                user_parts = [format_observation(observation)]
                history_text = format_history(history)
                if history_text:
                    user_parts.append(history_text)
                messages = [
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': '\n\n'.join(user_parts)},
                ]

                prompt_text = render_prompt(tokenizer, messages)
                if trainer.use_vllm:
                    rollout_output = generate_rollout_completions(trainer, [prompt_text])[0]
                else:
                    rollout_output = generate_rollout_completions_local(trainer, [prompt_text])[0]
                prompt_ids.extend(list(rollout_output['prompt_ids']))
                completion_ids.extend(list(rollout_output['completion_ids']))
                logprobs.extend([float(v) for v in rollout_output['logprobs']])

                completion_text = (rollout_output.get('text') or '').strip()
                if not sample_completion:
                    sample_completion = completion_text[:600]
                total_actions += 1
                action = parse_action_text(completion_text)
                if action is None:
                    step_rewards.append(-1.0)
                    history.append(
                        {
                            'action': completion_text or '(empty)',
                            'output': 'Invalid action format.',
                            'reward': -1.0,
                        }
                    )
                    continue

                valid_actions += 1
                tool_usage[action.tool] = tool_usage.get(action.tool, 0) + 1
                result = env.step(action)
                observation = result.observation
                reward = float(result.reward or 0.0)
                step_rewards.append(reward)
                tool_output = json.dumps(observation.tool_result or {}, default=str)
                history.append({'action': completion_text, 'output': tool_output, 'reward': reward})

            total_reward = sum(step_rewards) if step_rewards else -1.0
            mean_step_reward = (total_reward / len(step_rewards)) if step_rewards else 0.0
            min_step_reward = min(step_rewards) if step_rewards else 0.0
            max_step_reward = max(step_rewards) if step_rewards else 0.0
            valid_action_rate = (valid_actions / total_actions) if total_actions else 0.0
            mean_logprob = (sum(logprobs) / len(logprobs)) if logprobs else 0.0
            return {
                'prompt_ids': prompt_ids,
                'completion_ids': completion_ids,
                'logprobs': logprobs,
                'total_reward': total_reward,
                'mean_step_reward': mean_step_reward,
                'min_step_reward': min_step_reward,
                'max_step_reward': max_step_reward,
                'mean_logprob': mean_logprob,
                'completion_tokens': len(completion_ids),
                'prompt_tokens': len(prompt_ids),
                'steps': len(step_rewards),
                'scenario': scenario,
                'resolved': bool(result.done),
                'valid_actions': valid_actions,
                'total_actions': total_actions,
                'valid_action_rate': valid_action_rate,
                'tool_usage': tool_usage,
                'sample_completion': sample_completion,
            }
        except Exception as exc:
            last_exc = exc
            if not is_connection_error(exc) or attempt >= max_retries:
                raise
            wait_s = min(2 ** (attempt - 1), 8)
            print(
                f'Episode websocket dropped during scenario {scenario} '
                f'(attempt {attempt}/{max_retries}): {exc}. Retrying in {wait_s}s...'
            )
            time.sleep(wait_s)
        finally:
            try:
                env.close()
            except Exception:
                pass

    raise RuntimeError(f'Episode failed after {max_retries} reconnect attempts: {last_exc}')

if generate_rollout_completions is None:
    generate_rollout_completions = generate_rollout_completions_local
    if USE_VLLM:
        print(
            'Warning: trl.experimental.openenv is not in this trl build; vLLM rollouts fall back to '
            'local HF `model.generate` (slower, may not match vLLM logits). For real vLLM+OpenEnv, install '
            "a trl that ships `trl.experimental.openenv` or set USE_VLLM=False."
        )
    print('Rollout helpers ready (local gen; trl.experimental.openenv not in this trl build).')
else:
    print('Rollout helpers ready (trl.experimental.openenv).')

## 6. Smoke test the deployed Space

Hits the live KubeMedic environment to confirm the Space is reachable and tool calls return observations. Skip this cell by setting `SKIP_SMOKE_TEST = True` in the config cell.

In [ ]:
def smoke_test(env_url: str, scenario: str) -> None:
    env = make_env(env_url)
    try:
        result = env.reset(scenario=scenario)
        print(format_observation(result.observation)[:1200])
        step_result = env.step(KubemedicAction(tool='kubectl_get', args={'resource': 'pods', 'namespace': 'challenge'}))
        print('\n--- smoke tool call ---\n')
        print(format_observation(step_result.observation)[:1200])
        print('\nSmoke test passed.')
    finally:
        env.close()

if not SKIP_SMOKE_TEST:
    smoke_test(ENV_URL, SCENARIOS[0])
else:
    print('Skipping smoke test.')

## 7. Tokenizer and base model

Loads `unsloth/Qwen3-1.7B-unsloth-bnb-4bit` via unsloth's `FastLanguageModel` with `load_in_4bit=True` (QLoRA). Sets seeds, validates the `transformers` version, and applies a LoRA adapter to the standard Qwen attention + MLP projections.

In [ ]:
if SMOKE_ONLY:
    raise SystemExit('SMOKE_ONLY=True; stop here.')

if PER_DEVICE_TRAIN_BATCH_SIZE % NUM_GENERATIONS != 0:
    raise ValueError('PER_DEVICE_TRAIN_BATCH_SIZE must be divisible by NUM_GENERATIONS for GRPO.')

if Version(transformers.__version__) < Version('4.58.0'):
    raise RuntimeError(
        f"transformers>=4.58.0 is required for Qwen3.5 checkpoints (found {transformers.__version__}). "
        "Re-run the install cell and let it restart the kernel."
    )

set_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

use_vllm = torch.cuda.is_available() and USE_VLLM
if torch.cuda.is_available():
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    dtype = torch.float32

use_unsloth = bool(USE_UNSLOTH and UNSLOTH_AVAILABLE and torch.cuda.is_available())

LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
attn_impl = 'sdpa' if USE_FLASH_ATTENTION else 'eager'

if use_unsloth:
    if os.environ.get('HF_HUB_ENABLE_HF_TRANSFER') == '1':
        try:
            import hf_transfer  # noqa: F401
        except Exception:
            print('hf_transfer not importable; disabling HF_HUB_ENABLE_HF_TRANSFER for this run.')
            os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
    use_fast_model = ('gemma' in MODEL_ID.lower() and FastModel is not None)
    loader = FastModel if use_fast_model else FastLanguageModel
    from_pretrained_kwargs: dict[str, Any] = dict(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=dtype,
        load_in_4bit=USE_4BIT,
    )
    if use_vllm:
        from_pretrained_kwargs.update(
            fast_inference=True,
            max_lora_rank=LORA_RANK,
            gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
        )
    try:
        try:
            model, tokenizer = loader.from_pretrained(**from_pretrained_kwargs)
        except TypeError as kwarg_exc:
            if use_vllm:
                print(f'unsloth loader does not accept fast_inference kwargs ({kwarg_exc}); retrying without vLLM fast-inference.')
                for k in ('fast_inference', 'max_lora_rank', 'gpu_memory_utilization'):
                    from_pretrained_kwargs.pop(k, None)
                model, tokenizer = loader.from_pretrained(**from_pretrained_kwargs)
                use_vllm = False
            else:
                raise
        peft_kwargs: dict[str, Any] = dict(
            r=LORA_RANK,
            target_modules=LORA_TARGET_MODULES,
            lora_alpha=LORA_RANK * 2,
            lora_dropout=0.05,
            bias='none',
            use_gradient_checkpointing=('unsloth' if USE_GRADIENT_CHECKPOINTING else False),
            random_state=SEED,
            use_rslora=False,
            loftq_config=None,
        )
        if use_fast_model:
            peft_kwargs.update(
                finetune_vision_layers=False,
                finetune_language_layers=True,
                finetune_attention_modules=True,
                finetune_mlp_modules=True,
            )
        model = loader.get_peft_model(model, **peft_kwargs)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        peft_config = None  # unsloth already applied LoRA
        quant_config = None
        print('Loader    :', 'unsloth.FastModel' if use_fast_model else 'unsloth.FastLanguageModel')
        print('LoRA tgts :', LORA_TARGET_MODULES)
    except Exception as exc:
        print(f'unsloth loader failed at runtime ({exc}); falling back to standard transformers path.')
        use_unsloth = False

if not use_unsloth:
    fallback_model_ids = [MODEL_ID]
    if 'qwen3.5-2b' in MODEL_ID.lower():
        for _candidate in ('Qwen/Qwen3.5-2B', 'Qwen/Qwen3.5-2B-Base'):
            if _candidate not in fallback_model_ids:
                fallback_model_ids.append(_candidate)

    tokenizer = None
    resolved_model_id = None
    last_tokenizer_exc = None
    for _candidate in fallback_model_ids:
        try:
            tokenizer = AutoTokenizer.from_pretrained(
                _candidate,
                trust_remote_code=True,
                use_fast=False,
            )
            resolved_model_id = _candidate
            break
        except Exception as _tok_exc:
            last_tokenizer_exc = _tok_exc
            print(f'tokenizer load failed for {_candidate} ({_tok_exc})')

    if tokenizer is None:
        raise RuntimeError(f'Could not load tokenizer from any fallback id: {fallback_model_ids}. Last error: {last_tokenizer_exc}')
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if resolved_model_id and resolved_model_id != MODEL_ID:
        print(f'MODEL_ID fallback resolved: {MODEL_ID} -> {resolved_model_id}')
        MODEL_ID = resolved_model_id

    quant_config = None
    if torch.cuda.is_available() and USE_4BIT:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=dtype,
            bnb_4bit_use_double_quant=True,
        )

    model_kwargs: dict[str, Any] = {
        'torch_dtype': dtype,
        'quantization_config': quant_config,
        'attn_implementation': attn_impl,
    }
    if torch.cuda.is_available():
        model_kwargs['device_map'] = 'auto'

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        **model_kwargs,
    )

    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=infer_lora_target_modules(model),
    )
    print('Loader    : transformers.AutoModelForCausalLM')
    print('LoRA tgts :', peft_config.target_modules)

print('use_vllm  :', use_vllm)
print('use_unsloth:', use_unsloth)
print('attn impl :', attn_impl)
print('dtype     :', dtype)
print('4-bit     :', USE_4BIT and torch.cuda.is_available())
print('grad ckpt :', USE_GRADIENT_CHECKPOINTING)

## 8. GRPO trainer setup and training

Builds the train/eval datasets, the GRPO config, the rollout function (which logs episodes to disk), and runs `trainer.train()`.

In [ ]:
train_dataset = build_dataset(TRAIN_EPISODES)
eval_dataset = build_dataset(EVAL_EPISODES)

_trainer_grad_ckpt = bool(USE_GRADIENT_CHECKPOINTING and not use_unsloth)

grpo_config_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    num_train_epochs=NUM_EPOCHS,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    save_total_limit=2,
    bf16=(dtype == torch.bfloat16),
    fp16=(dtype == torch.float16),
    tf32=torch.cuda.is_available(),
    gradient_checkpointing=_trainer_grad_ckpt,
    remove_unused_columns=False,
    report_to='none',  # local-only run; metrics go to OUTPUT_DIR
    use_vllm=use_vllm,
    vllm_mode=VLLM_MODE,
    vllm_gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    log_completions=True,
    max_tool_calling_iterations=1,
    seed=SEED,
    optim='adamw_torch_fused',
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

try:
    grpo_config = GRPOConfig(**grpo_config_kwargs)
except TypeError as exc:
    print(f'GRPOConfig rejected a kwarg ({exc}); retrying with the safe subset.')
    for k in ('tf32', 'optim', 'dataloader_num_workers', 'dataloader_pin_memory'):
        grpo_config_kwargs.pop(k, None)
    grpo_config = GRPOConfig(**grpo_config_kwargs)

# ---- Local metrics persistence ---------------------------------------------
# Three files, all in OUTPUT_DIR, all written incrementally during training so
# you can tail them mid-run and still get something useful if training crashes:
#
#   episode_rewards.csv   slim per-episode summary, easy to load with pandas
#   episode_metrics.jsonl rich per-episode JSON (tool usage, logprobs, etc.)
#   agent_transcripts.jsonl full per-episode trajectories (prompt_ids etc.)

reward_log_path = OUTPUT_DIR / 'episode_rewards.csv'
metrics_log_path = OUTPUT_DIR / 'episode_metrics.jsonl'
transcript_path = OUTPUT_DIR / 'agent_transcripts.jsonl'

with reward_log_path.open('w', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow([
        'episode', 'scenario', 'total_reward', 'mean_step_reward',
        'min_step_reward', 'max_step_reward', 'steps', 'resolved',
        'valid_action_rate', 'valid_actions', 'total_actions',
        'completion_tokens', 'prompt_tokens', 'mean_logprob',
        'running_mean_reward', 'running_resolution_rate',
    ])
metrics_log_path.write_text('')  # truncate any prior run's JSONL
transcript_path.write_text('')

episode_counter = {'value': 0}

ROLL_WINDOW = 32
recent_rewards: list[float] = []
recent_resolved: list[int] = []
recent_steps: list[int] = []
recent_valid_rate: list[float] = []
scenario_rewards: dict[str, list[float]] = {s: [] for s in SCENARIOS}


def _running_mean(values, window: int) -> float:
    if not values:
        return 0.0
    tail = values[-window:]
    return sum(tail) / len(tail)


def rollout_func(prompts: list[str], trainer: GRPOTrainer) -> dict[str, list]:
    prompt_count = len(prompts)
    episode_prompt_ids: list[list[int]] = []
    episode_completion_ids: list[list[int]] = []
    episode_logprobs: list[list[float]] = []
    total_rewards: list[float] = []
    steps: list[int] = []
    scenario_names: list[str] = []
    resolved_flags: list[bool] = []

    for start in range(0, prompt_count, NUM_GENERATIONS):
        scenario = random.choice(SCENARIOS)
        group_size = min(NUM_GENERATIONS, prompt_count - start)
        for _ in range(group_size):
            episode = rollout_once(
                trainer=trainer,
                env_url=ENV_URL,
                tokenizer=tokenizer,
                scenario=scenario,
                max_turns=MAX_TURNS,
                max_total_tokens=4096,
            )
            episode_prompt_ids.append(episode['prompt_ids'])
            episode_completion_ids.append(episode['completion_ids'])
            episode_logprobs.append(episode['logprobs'])
            total_rewards.append(float(episode['total_reward']))
            steps.append(int(episode['steps']))
            scenario_names.append(episode['scenario'])
            resolved_flags.append(bool(episode['resolved']))

            episode_counter['value'] += 1
            recent_rewards.append(float(episode['total_reward']))
            recent_resolved.append(int(bool(episode['resolved'])))
            recent_steps.append(int(episode['steps']))
            recent_valid_rate.append(float(episode['valid_action_rate']))
            scenario_rewards.setdefault(episode['scenario'], []).append(
                float(episode['total_reward'])
            )

            running_reward = _running_mean(recent_rewards, ROLL_WINDOW)
            running_resolution = _running_mean(
                [float(v) for v in recent_resolved], ROLL_WINDOW
            )

            with reward_log_path.open('a', newline='') as handle:
                writer = csv.writer(handle)
                writer.writerow([
                    episode_counter['value'],
                    episode['scenario'],
                    episode['total_reward'],
                    episode['mean_step_reward'],
                    episode['min_step_reward'],
                    episode['max_step_reward'],
                    episode['steps'],
                    int(bool(episode['resolved'])),
                    episode['valid_action_rate'],
                    episode['valid_actions'],
                    episode['total_actions'],
                    episode['completion_tokens'],
                    episode['prompt_tokens'],
                    episode['mean_logprob'],
                    running_reward,
                    running_resolution,
                ])

            tool_usage = episode.get('tool_usage') or {}
            metrics_payload = {
                'episode': episode_counter['value'],
                'scenario': episode['scenario'],
                'total_reward': float(episode['total_reward']),
                'mean_step_reward': float(episode['mean_step_reward']),
                'min_step_reward': float(episode['min_step_reward']),
                'max_step_reward': float(episode['max_step_reward']),
                'mean_logprob': float(episode['mean_logprob']),
                'steps': int(episode['steps']),
                'resolved': bool(episode['resolved']),
                'valid_action_rate': float(episode['valid_action_rate']),
                'valid_actions': int(episode['valid_actions']),
                'total_actions': int(episode['total_actions']),
                'completion_tokens': int(episode['completion_tokens']),
                'prompt_tokens': int(episode['prompt_tokens']),
                'tool_usage': {k: int(v) for k, v in tool_usage.items()},
                'sample_completion': episode.get('sample_completion', ''),
                'running_mean_reward': running_reward,
                'running_resolution_rate': running_resolution,
            }
            with metrics_log_path.open('a') as handle:
                handle.write(json.dumps(metrics_payload) + '\n')
            with transcript_path.open('a') as handle:
                handle.write(json.dumps(episode) + '\n')

    return {
        'prompt_ids': episode_prompt_ids,
        'completion_ids': episode_completion_ids,
        'logprobs': episode_logprobs,
        'total_reward': total_rewards,
        'steps': steps,
        'scenario': scenario_names,
        'resolved': resolved_flags,
    }


os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')

import sys as _sys
_nb_stdout, _nb_stderr = _sys.stdout, _sys.stderr
if hasattr(_sys.__stdout__, 'fileno') and hasattr(_sys.__stderr__, 'fileno'):
    _sys.stdout, _sys.stderr = _sys.__stdout__, _sys.__stderr__
try:
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=reward_total,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=grpo_config,
        rollout_func=rollout_func,
        peft_config=peft_config,
    )
finally:
    _sys.stdout, _sys.stderr = _nb_stdout, _nb_stderr

trainer.train()

## 9. Save artifacts and plots

Saves the LoRA adapter + tokenizer, writes the reward CSV plot, the trainer log history, and a JSON training summary.

In [ ]:
# Save LoRA adapter + tokenizer.
adapter_dir = OUTPUT_DIR / 'adapter'
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

# Persist trainer's own per-step log so we can plot loss/grad_norm/lr.
log_history_path = OUTPUT_DIR / 'trainer_log_history.csv'
log_history = pd.DataFrame(trainer.state.log_history)
log_history.to_csv(log_history_path, index=False)


def _rolling(series: pd.Series, window: int = 10) -> pd.Series:
    return series.rolling(window=min(window, max(len(series), 1)), min_periods=1).mean()


def plot_reward_curve(rows: pd.DataFrame, out_path: Path) -> None:
    """Per-episode total reward + rolling mean."""
    if rows.empty:
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(rows['episode'], rows['total_reward'],
            marker='o', alpha=0.35, label='episode reward')
    ax.plot(rows['episode'], _rolling(rows['total_reward']),
            linewidth=2, label=f'rolling mean (n={min(10, len(rows))})')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Total reward')
    ax.set_title('Reward curve')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)


def plot_resolution_rate(rows: pd.DataFrame, out_path: Path) -> None:
    """Cumulative + rolling resolution rate."""
    if rows.empty:
        return
    cumulative = rows['resolved'].expanding().mean()
    rolling = _rolling(rows['resolved'].astype(float))
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(rows['episode'], cumulative, label='cumulative resolution rate', linewidth=2)
    ax.plot(rows['episode'], rolling, label='rolling resolution rate', linewidth=2, alpha=0.7)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Resolution rate')
    ax.set_ylim(-0.05, 1.05)
    ax.set_title('Resolution rate over episodes')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)


def plot_action_quality(rows: pd.DataFrame, out_path: Path) -> None:
    """Steps per episode + valid-action-rate, dual axis."""
    if rows.empty:
        return
    fig, ax_left = plt.subplots(figsize=(12, 5))
    ax_left.plot(rows['episode'], _rolling(rows['steps'].astype(float)),
                 label='rolling mean steps', color='tab:blue', linewidth=2)
    ax_left.set_xlabel('Episode')
    ax_left.set_ylabel('Steps per episode', color='tab:blue')
    ax_left.tick_params(axis='y', labelcolor='tab:blue')
    ax_right = ax_left.twinx()
    ax_right.plot(rows['episode'], _rolling(rows['valid_action_rate']),
                  label='rolling valid-action rate', color='tab:orange', linewidth=2)
    ax_right.set_ylabel('Valid-action rate', color='tab:orange')
    ax_right.set_ylim(-0.05, 1.05)
    ax_right.tick_params(axis='y', labelcolor='tab:orange')
    ax_left.set_title('Action quality')
    ax_left.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)


def plot_per_scenario(rows: pd.DataFrame, out_path: Path) -> None:
    """Reward distribution per scenario (boxplot) + per-scenario count."""
    if rows.empty or 'scenario' not in rows.columns:
        return
    fig, (ax_box, ax_bar) = plt.subplots(1, 2, figsize=(14, 5))
    sns.boxplot(data=rows, x='scenario', y='total_reward', ax=ax_box,
                order=sorted(rows['scenario'].unique()))
    ax_box.set_title('Reward distribution per scenario')
    ax_box.set_xlabel('Scenario')
    ax_box.set_ylabel('Total reward')
    ax_box.tick_params(axis='x', rotation=30)
    ax_box.grid(True, alpha=0.3, axis='y')

    counts = rows.groupby('scenario')['resolved'].agg(['mean', 'count'])
    counts = counts.sort_index()
    bars = ax_bar.bar(counts.index, counts['mean'], color='tab:green', alpha=0.7)
    for bar, n in zip(bars, counts['count']):
        ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                    f'n={int(n)}', ha='center', va='bottom', fontsize=9)
    ax_bar.set_title('Resolution rate per scenario')
    ax_bar.set_ylabel('Resolved fraction')
    ax_bar.set_ylim(0, 1.1)
    ax_bar.tick_params(axis='x', rotation=30)
    ax_bar.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)


def plot_tool_usage(metrics_path: Path, out_path: Path) -> None:
    """Bar chart of total tool calls aggregated across all episodes."""
    if not metrics_path.exists():
        return
    counts: dict[str, int] = {}
    for line in metrics_path.read_text().splitlines():
        if not line.strip():
            continue
        try:
            payload = json.loads(line)
        except json.JSONDecodeError:
            continue
        for tool, count in (payload.get('tool_usage') or {}).items():
            counts[tool] = counts.get(tool, 0) + int(count)
    if not counts:
        return
    items = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)
    names = [k for k, _ in items]
    values = [v for _, v in items]
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(names, values, color='tab:purple', alpha=0.75)
    ax.set_title('Tool usage (total calls across all episodes)')
    ax.set_ylabel('Calls')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)


def plot_trainer_metrics(log_history: pd.DataFrame, out_path: Path) -> None:
    """Trainer's own loss / grad_norm / learning_rate over training steps."""
    if log_history.empty:
        return
    cols = [c for c in ['loss', 'grad_norm', 'learning_rate'] if c in log_history.columns]
    if not cols or 'step' not in log_history.columns:
        return
    fig, axes = plt.subplots(1, len(cols), figsize=(6 * len(cols), 4))
    if len(cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cols):
        subset = log_history.dropna(subset=[col])
        if subset.empty:
            continue
        sns.lineplot(data=subset, x='step', y=col, ax=ax)
        ax.set_title(col)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)


# Generate every plot.
reward_plot_path = OUTPUT_DIR / 'reward_plot.png'
resolution_plot_path = OUTPUT_DIR / 'resolution_plot.png'
action_plot_path = OUTPUT_DIR / 'action_quality_plot.png'
scenario_plot_path = OUTPUT_DIR / 'per_scenario_plot.png'
tool_usage_plot_path = OUTPUT_DIR / 'tool_usage_plot.png'
trainer_metrics_path = OUTPUT_DIR / 'trainer_metrics.png'
dashboard_path = OUTPUT_DIR / 'dashboard.png'

rows = pd.read_csv(reward_log_path) if reward_log_path.exists() else pd.DataFrame()
plot_reward_curve(rows, reward_plot_path)
plot_resolution_rate(rows, resolution_plot_path)
plot_action_quality(rows, action_plot_path)
plot_per_scenario(rows, scenario_plot_path)
plot_tool_usage(metrics_log_path, tool_usage_plot_path)
plot_trainer_metrics(log_history, trainer_metrics_path)


def plot_dashboard(rows: pd.DataFrame, log_history: pd.DataFrame, out_path: Path) -> None:
    """One PNG with the four most-useful charts side-by-side."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    ax_reward, ax_res, ax_action, ax_loss = axes.flat
    if not rows.empty:
        ax_reward.plot(rows['episode'], rows['total_reward'],
                       marker='o', alpha=0.3, label='episode')
        ax_reward.plot(rows['episode'], _rolling(rows['total_reward']),
                       linewidth=2, label='rolling')
        ax_reward.set_title('Reward')
        ax_reward.set_xlabel('Episode')
        ax_reward.legend()
        ax_reward.grid(True, alpha=0.3)

        cumulative = rows['resolved'].expanding().mean()
        ax_res.plot(rows['episode'], cumulative, linewidth=2, label='cumulative')
        ax_res.plot(rows['episode'], _rolling(rows['resolved'].astype(float)),
                    linewidth=2, alpha=0.7, label='rolling')
        ax_res.set_title('Resolution rate')
        ax_res.set_xlabel('Episode')
        ax_res.set_ylim(-0.05, 1.05)
        ax_res.legend()
        ax_res.grid(True, alpha=0.3)

        ax_action.plot(rows['episode'], _rolling(rows['valid_action_rate']),
                       linewidth=2, label='valid-action rate')
        ax_action.set_title('Action quality')
        ax_action.set_xlabel('Episode')
        ax_action.set_ylim(-0.05, 1.05)
        ax_action.legend()
        ax_action.grid(True, alpha=0.3)
    if not log_history.empty and 'loss' in log_history.columns and 'step' in log_history.columns:
        loss_subset = log_history.dropna(subset=['loss'])
        if not loss_subset.empty:
            ax_loss.plot(loss_subset['step'], loss_subset['loss'], linewidth=2)
            ax_loss.set_title('Trainer loss')
            ax_loss.set_xlabel('Step')
            ax_loss.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)


plot_dashboard(rows, log_history, dashboard_path)

# Run summary.
summary_path = OUTPUT_DIR / 'training_summary.json'
final_metrics = {}
if not rows.empty:
    final_metrics = {
        'episodes_run': int(len(rows)),
        'final_total_reward': float(rows['total_reward'].iloc[-1]),
        'mean_total_reward': float(rows['total_reward'].mean()),
        'best_total_reward': float(rows['total_reward'].max()),
        'final_running_mean_reward': float(
            rows.get('running_mean_reward', rows['total_reward']).iloc[-1]
        ),
        'cumulative_resolution_rate': float(rows['resolved'].mean()),
        'mean_steps_per_episode': float(rows['steps'].mean()),
        'mean_valid_action_rate': float(rows['valid_action_rate'].mean()),
        'per_scenario_resolution_rate': rows.groupby('scenario')['resolved']
            .mean().to_dict(),
        'per_scenario_episode_count': rows.groupby('scenario')['resolved']
            .count().to_dict(),
    }

summary = {
    'model_id': MODEL_ID,
    'env_url': ENV_URL,
    'output_dir': str(OUTPUT_DIR.resolve()),
    'adapter_dir': str(adapter_dir.resolve()),
    'config': {
        'train_episodes': TRAIN_EPISODES,
        'eval_episodes': EVAL_EPISODES,
        'num_epochs': NUM_EPOCHS,
        'num_generations': NUM_GENERATIONS,
        'per_device_train_batch_size': PER_DEVICE_TRAIN_BATCH_SIZE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
        'learning_rate': LEARNING_RATE,
        'max_completion_length': MAX_COMPLETION_LENGTH,
        'max_turns': MAX_TURNS,
        'use_4bit': USE_4BIT,
        'use_unsloth': use_unsloth,
        'use_vllm': use_vllm,
        'use_gradient_checkpointing': bool(USE_GRADIENT_CHECKPOINTING),
        'lora_rank': LORA_RANK,
        'seed': SEED,
        'scenarios': SCENARIOS,
    },
    'final_metrics': final_metrics,
    'artifacts': {
        'episode_rewards_csv': str(reward_log_path.relative_to(OUTPUT_DIR)),
        'episode_metrics_jsonl': str(metrics_log_path.relative_to(OUTPUT_DIR)),
        'agent_transcripts_jsonl': str(transcript_path.relative_to(OUTPUT_DIR)),
        'trainer_log_history_csv': str(log_history_path.relative_to(OUTPUT_DIR)),
        'reward_plot': str(reward_plot_path.relative_to(OUTPUT_DIR)),
        'resolution_plot': str(resolution_plot_path.relative_to(OUTPUT_DIR)),
        'action_quality_plot': str(action_plot_path.relative_to(OUTPUT_DIR)),
        'per_scenario_plot': str(scenario_plot_path.relative_to(OUTPUT_DIR)),
        'tool_usage_plot': str(tool_usage_plot_path.relative_to(OUTPUT_DIR)),
        'trainer_metrics_plot': str(trainer_metrics_path.relative_to(OUTPUT_DIR)),
        'dashboard': str(dashboard_path.relative_to(OUTPUT_DIR)),
    },
}
summary_path.write_text(json.dumps(summary, indent=2, default=str))

print('Wrote artifacts under', OUTPUT_DIR.resolve())
for name, rel in summary['artifacts'].items():
    target = OUTPUT_DIR / rel
    flag = ' ' if target.exists() else '!'
    print(f'  [{flag}] {name:30s} {rel}')

## 10. Inspect results

Read back the reward CSV, summary JSON, and plots saved above.

In [ ]:
from IPython.display import Image, Markdown, display

reward_csv = OUTPUT_DIR / 'episode_rewards.csv'
summary_json = OUTPUT_DIR / 'training_summary.json'

if reward_csv.exists():
    display(Markdown('### Last 10 episodes'))
    display(pd.read_csv(reward_csv).tail(10))
else:
    raise FileNotFoundError(
        f'{reward_csv} was not created. Check the training cell output above for the real failure.'
    )

if summary_json.exists():
    summary = json.loads(summary_json.read_text())
    display(Markdown('### Run summary'))
    print(json.dumps(summary['config'], indent=2))
    display(Markdown('### Final metrics'))
    print(json.dumps(summary['final_metrics'], indent=2, default=str))

# Display every plot we produced. Order is: dashboard first (overview), then
# the focused per-axis plots so you can drill into specifics.
plots_to_show = [
    ('Dashboard', OUTPUT_DIR / 'dashboard.png'),
    ('Reward curve', OUTPUT_DIR / 'reward_plot.png'),
    ('Resolution rate', OUTPUT_DIR / 'resolution_plot.png'),
    ('Action quality', OUTPUT_DIR / 'action_quality_plot.png'),
    ('Per-scenario', OUTPUT_DIR / 'per_scenario_plot.png'),
    ('Tool usage', OUTPUT_DIR / 'tool_usage_plot.png'),
    ('Trainer metrics', OUTPUT_DIR / 'trainer_metrics.png'),
]
for title, path in plots_to_show:
    if path.exists():
        display(Markdown(f'### {title}'))
        display(Image(filename=str(path)))